Questo è il Notebook relativo alla task 4 del progetto.

In [ ]:
#Librerie
import numpy as np
from scipy.sparse import csc_matrix, coo_matrix
from scipy.sparse.linalg import spsolve_triangular
from sksparse.cholmod import cholesky

#funzione che legge le matrici in formato COO e restituisce la matrice in formato CSC
def read_coo_to_csc(a_filename):
    # Caricamento di A e conversione in CSC
    data_A = np.loadtxt(a_filename)
    rows = data_A[:, 0].astype(int)
    cols = data_A[:, 1].astype(int)
    vals = data_A[:, 2]
    A_coo = coo_matrix((vals, (rows, cols)))
    A_csc = A_coo.tocsc()
    return A_csc

A_orig_CSC = read_coo_to_csc("A_orig.txt")
A_reordered_CSC = read_coo_to_csc("A_reordered.txt")

print("=== MATRICE ORIGINALE ===")
print(f"Dimensione : {A_orig_CSC.shape}") #mi dice la dimensione della matrice n x m
print(f"NNZ        : {A_orig_CSC.nnz}\n") #mi dice il numero di elementi non nulli

print("=== MATRICE RIORDINATA ===")
print(f"Dimensione : {A_reordered_CSC.shape}")
print(f"NNZ        : {A_reordered_CSC.nnz}\n")

A_orig = -A_orig_CSC #rendo la matrice definita positiva
A_reordered = -A_reordered_CSC #rendo la matrice definita positiva

#funzione che calcola la fattorizzazione di Cholesky di una matrice
def my_cholesky(A):
    try:
        factor = cholesky(A, order="natural")
    except TypeError:
        try:
            factor = cholesky(A, ordering_method="natural")
        except TypeError:
            factor = cholesky(A)

    # 'factor' e' una TUPLA, quindi usiamo [0]
    return csc_matrix(factor[0])

L_orig = my_cholesky(A_orig)
L_reordered = my_cholesky(A_reordered)

#funzione che legge il termine noto b
def read_rhs(rhs_filename):
    data = np.loadtxt(rhs_filename)
    return data

b_orig = -read_rhs("rhs_orig.txt")
b_reordered = -read_rhs("rhs_reordered.txt")

#funzione che risolve il sistema lineare Ax = b usando la fattorizzazione di Cholesky
def solve_cholesky(L, b):
    # Risolvo Ly = b
    y = spsolve_triangular(L, b, lower=True)
    # Risolvo L^Tu = y
    u = spsolve_triangular(L.T.tocsc(), y, lower=False)
    return u

# Calcolo delle soluzioni
u_orig = solve_cholesky(L_orig, b_orig)
u_reordered = solve_cholesky(L_reordered, b_reordered)



# Stampa dei risultati 
print("=== SISTEMA ORIGINALE ===")
print(f"Fill-in NNZ(L)   : {L_orig.nnz}\n")

print("=== SISTEMA RIORDINATO ===")
print(f"Fill-in NNZ(L)   : {L_reordered.nnz}\n")

#ATTENZIONE: i valori di u_orig e u_reordered sono gli stessi, ma l'ordine degli elementi è diverso. In totale ho N^2 elementi
print("Primi 10 elementi della soluzione:", u_orig[:10])
print("Primi 10 elementi della soluzione riordinata:", u_reordered[:10])

import time

# --- SISTEMA ORIGINALE ---
#  Tempo di fattorizzazione
t0 = time.perf_counter()
L_orig = my_cholesky(A_orig)
t_fact_orig = time.perf_counter() - t0

#  Tempo di risoluzione dei sistemi triangolari
t0 = time.perf_counter()
u_orig = solve_cholesky(L_orig, b_orig)
t_solve_orig = time.perf_counter() - t0

# --- SISTEMA RIORDINATO ---
t0 = time.perf_counter()
L_reordered = my_cholesky(A_reordered)
t_fact_reord = time.perf_counter() - t0

t0 = time.perf_counter()
u_reordered = solve_cholesky(L_reordered, b_reordered)
t_solve_reord = time.perf_counter() - t0

print("=== PRESTAZIONI (TEMPI IN SECONDI) ===")
print(f"Originale  -> Fattorizzazione: {t_fact_orig:.6f} s | Soluzione: {t_solve_orig:.6f} s | Totale: {t_fact_orig + t_solve_orig:.6f} s")
print(f"Riordinato -> Fattorizzazione: {t_fact_reord:.6f} s | Soluzione: {t_solve_reord:.6f} s | Totale: {t_fact_reord + t_solve_reord:.6f} s")
print(f"Speedup Fattorizzazione: {t_fact_orig / t_fact_reord:.2f}x")
